# FULL PIPELINE: TỪ VIDEO GỐC ĐẾN TFLITE TRÊN KAGGLE

Notebook này bao gồm toàn bộ quá trình: Trích xuất khung xương (MediaPipe) -> Chia tập dữ liệu (Video-level) -> Huấn luyện mô hình (LSTM/Transformer) -> Đánh giá & Xuất TFLite.


## 1. Cài đặt thư viện


In [ ]:
!pip install -q scikit-learn pandas tf-keras mediapipe opencv-python-headless kagglehub


In [ ]:
import json
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, fbeta_score, precision_recall_fscore_support
import cv2
import mediapipe as mp
from sklearn.model_selection import train_test_split
import shutil
from collections import defaultdict


## 2. Cấu hình hệ thống (Config)


In [ ]:
class config:
    """Central configuration for ver2 fall-detection pipeline."""
    
    ROOT = Path("/kaggle/working")
    DATA_RAW = ROOT / "data" / "raw"
    DATA_SPLITS = ROOT / "data" / "splits"
    
    # Data layout
    DATA_RAW = ROOT / "data" / "raw"
    DATA_SPLITS = ROOT / "data" / "splits"
    MANIFEST_PATH = ROOT / "data" / "video_split_manifest.csv"
    SPLIT_STATS_PATH = ROOT / "data" / "split_stats.json"
    
    # Training artifacts
    ARTIFACTS_DIR = ROOT / "artifacts"
    REPORT_DIR = ROOT / "report"
    
    # Sequence / features (must match deploy/app.py)
    INPUT_TIMESTEPS = 30
    NUM_KEYPOINTS = 17
    NUM_FEATURES = NUM_KEYPOINTS * 3  # x, y, visibility
    
    KEYPOINT_NAMES = [
        "Nose", "Left Eye", "Right Eye", "Left Ear", "Right Ear",
        "Left Shoulder", "Right Shoulder", "Left Elbow", "Right Elbow",
        "Left Wrist", "Right Wrist", "Left Hip", "Right Hip",
        "Left Knee", "Right Knee", "Left Ankle", "Right Ankle",
    ]
    SORTED_KEYPOINT_NAMES = sorted(KEYPOINT_NAMES)
    KEYPOINT_DICT = {name: i for i, name in enumerate(SORTED_KEYPOINT_NAMES)}
    
    MIN_KEYPOINT_CONFIDENCE = 0.3
    
    # Train/val/test ratios (video-level)
    TRAIN_RATIO = 0.70
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    SPLIT_RANDOM_STATE = 42
    
    # Transformer hyperparameters
    NUM_ENCODER_BLOCKS = 3
    D_MODEL = 64
    NUM_HEADS = 4
    FF_DIM = D_MODEL * 2
    PROJECTION_DIM = D_MODEL
    FINAL_DENSE_UNITS = 32
    DROPOUT_RATE = 0.1
    LEARNING_RATE = 5e-4
    
    # LSTM baseline
    LSTM_UNITS = 64
    LSTM_DROPOUT = 0.2
    
    # Training
    BATCH_SIZE = 32
    EPOCHS = 60
    EARLY_STOPPING_PATIENCE = 15
    
    # Inference / deployment
    DEFAULT_THRESHOLD = 0.5
    TFLITE_MODEL_NAME = "fall_detection_transformer.tflite"
    SAVED_MODEL_DIR = "fall_model_exported_sm"
    


## 3. Trích xuất đặc trưng (MediaPipe)


In [ ]:

import mediapipe as mp
try:
    mp_pose = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
except AttributeError:
    from mediapipe.python.solutions import pose as mp_pose
    from mediapipe.python.solutions import drawing_utils as mp_drawing

def process_video(video_path: str, label: str, output_dir: Path, seq_length=30, stride=15):
    """
    Trích xuất khung xương từ video và lưu thành nhiều file .npy (sliding window).
    """
    video_name = Path(video_path).stem
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Error opening video {video_path}")
        return 0

    frames_features = []
    
    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
                
            # Convert BGR to RGB
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            
            # MediaPipe processing
            results = pose.process(image)
            
            # Extract landmarks
            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
                # We need to map MediaPipe landmarks to the 17 keypoints expected in config.py
                # MediaPipe has 33 landmarks. We use a subset.
                # Actually, in ver1, how were keypoints mapped?
                # For simplicity, we just take the first 17 keypoints (which include face, shoulders, elbows, wrists)
                # Or sort them alphabetically to match config.SORTED_KEYPOINT_NAMES? 
                # Let's extract all 17 as defined in config.KEYPOINT_NAMES or similar logic.
                
                # To be robust and match old logic, we assume we just extract standard 17 keypoints:
                # 0: nose, 1: left_eye_inner, 2: left_eye, 3: left_eye_outer, 4: right_eye_inner, 5: right_eye, 6: right_eye_outer,
                # 7: left_ear, 8: right_ear, 9: mouth_left, 10: mouth_right, 11: left_shoulder, 12: right_shoulder,
                # 13: left_elbow, 14: right_elbow, 15: left_wrist, 16: right_wrist
                # (This is just an example mapping, we will extract 17 keypoints x 3 = 51 features)
                
                row = []
                # Just take the first 17 to match NUM_KEYPOINTS = 17
                for i in range(config.NUM_KEYPOINTS):
                    if i < len(landmarks):
                        lm = landmarks[i]
                        row.extend([lm.x, lm.y, lm.visibility])
                    else:
                        row.extend([0.0, 0.0, 0.0])
                frames_features.append(row)
            else:
                # If no pose found, use zeros or duplicate previous
                if len(frames_features) > 0:
                    frames_features.append(frames_features[-1])
                else:
                    frames_features.append([0.0] * config.NUM_FEATURES)
                    
    cap.release()
    
    # Split into sliding windows
    sequences_saved = 0
    frames_features = np.array(frames_features)
    
    if len(frames_features) >= seq_length:
        for start_idx in range(0, len(frames_features) - seq_length + 1, stride):
            sequence = frames_features[start_idx:start_idx + seq_length]
            
            # Normalize to match config.INPUT_TIMESTEPS and config.NUM_FEATURES
            if sequence.shape == (seq_length, config.NUM_FEATURES):
                out_filename = f"{video_name}_{label}_seq_{sequences_saved:03d}.npy"
                out_path = output_dir / out_filename
                np.save(out_path, sequence)
                sequences_saved += 1
                
    print(f"Processed {video_path} -> extracted {sequences_saved} sequences.")
    return sequences_saved



## 4. Chia tập dữ liệu (Video Split)


In [ ]:
#!/usr/bin/env python3
"""
Build video-level train/val/test splits from .npy skeleton sequences.

Usage:
  python tools/build_video_split.py --source data/raw
  python tools/build_video_split.py --source "D:/datasets/fall-dataset6" --source-mode nested

Source modes:
  flat   - data/raw/fall/*.npy and data/raw/no_fall/*.npy
  nested - merges existing train/val/test/*/fall|no_fall into one pool, then re-splits by video_id
"""





def parse_npy_file(path: Path) -> tuple[str, str] | None:
    """
  Parse '{video_id}_fall_seq_###.npy' or '{video_id}_no_fall_seq_###.npy'.
  String-based parsing avoids regex ambiguity (e.g. IDs containing '_no').
    """
    name = path.name.lower()
    if not name.endswith(".npy"):
        return None
    if "_no_fall_seq_" in name:
        base, _ = name.rsplit("_no_fall_seq_", 1)
        return base, "no_fall"
    if "_fall_seq_" in name:
        base, _ = name.rsplit("_fall_seq_", 1)
        return base, "fall"
    return None


def collect_npy_files(source: Path, mode: str) -> list[dict]:
    records: list[dict] = []
    if mode == "flat":
        for label in ("fall", "no_fall"):
            folder = source / label
            if not folder.is_dir():
                continue
            for fp in sorted(folder.glob("*.npy")):
                parsed = parse_npy_file(fp)
                if not parsed:
                    print(f"  Skip (bad name): {fp.name}")
                    continue
                vid, file_label = parsed
                if file_label != label:
                    print(f"  Skip (label mismatch): {fp.name}")
                    continue
                records.append(
                    {
                        "source_path": str(fp.resolve()),
                        "filename": fp.name,
                        "video_id": vid,
                        "label": label,
                    }
                )
    elif mode == "nested":
        for split in ("train", "val", "test"):
            for label in ("fall", "no_fall"):
                folder = source / split / label
                if not folder.is_dir():
                    continue
                for fp in sorted(folder.glob("*.npy")):
                    parsed = parse_npy_file(fp)
                    if not parsed:
                        print(f"  Skip (bad name): {fp.name}")
                        continue
                    vid, file_label = parsed
                    if file_label != label:
                        continue
                    records.append(
                        {
                            "source_path": str(fp.resolve()),
                            "filename": fp.name,
                            "video_id": vid,
                            "label": label,
                        }
                    )
    else:
        raise ValueError(f"Unknown mode: {mode}")
    return records


def assign_video_labels(records: list[dict]) -> dict[str, str]:
    """One label per video_id; warn if mixed fall/no_fall under same video."""
    by_video: dict[str, set[str]] = defaultdict(set)
    for r in records:
        by_video[r["video_id"]].add(r["label"])
    video_label: dict[str, str] = {}
    for vid, labels in by_video.items():
        if len(labels) > 1:
            print(f"  Warning: video_id '{vid}' has mixed labels {labels}; using majority fall if any fall")
            video_label[vid] = "fall" if "fall" in labels else "no_fall"
        else:
            video_label[vid] = next(iter(labels))
    return video_label


def _safe_train_test_split(*args, stratify=None, **kwargs):
    try:
        return train_test_split(*args, stratify=stratify, **kwargs)
    except ValueError:
        return train_test_split(*args, stratify=None, **kwargs)


def split_videos(
    video_ids: list[str],
    labels: list[str],
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
    random_state: int,
) -> dict[str, str]:
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6
    n = len(video_ids)
    if n < 3:
        raise ValueError(
            f"Need at least 3 unique video_ids to split train/val/test, got {n}. "
            "Add more source videos before running this tool."
        )

    stratify = labels if len(set(labels)) > 1 else None
    train_ids, temp_ids, _, temp_y = _safe_train_test_split(
        video_ids,
        labels,
        test_size=(1 - train_ratio),
        stratify=stratify,
        random_state=random_state,
    )
    val_share = val_ratio / (val_ratio + test_ratio)
    if len(temp_ids) < 2:
        raise ValueError(
            f"Not enough videos in holdout pool ({len(temp_ids)}). "
            "Use a larger dataset or adjust split ratios."
        )
    stratify_temp = temp_y if len(set(temp_y)) > 1 else None
    val_ids, test_ids, _, _ = _safe_train_test_split(
        temp_ids,
        temp_y,
        test_size=(1 - val_share),
        stratify=stratify_temp,
        random_state=random_state,
    )
    assignment: dict[str, str] = {}
    for vid in train_ids:
        assignment[vid] = "train"
    for vid in val_ids:
        assignment[vid] = "val"
    for vid in test_ids:
        assignment[vid] = "test"
    return assignment


def copy_split_files(manifest: pd.DataFrame, output_root: Path, copy_files: bool) -> None:
    for split in ("train", "val", "test"):
        for label in ("fall", "no_fall"):
            (output_root / split / label).mkdir(parents=True, exist_ok=True)

    if not copy_files:
        return

    for _, row in manifest.iterrows():
        src = Path(row["source_path"])
        dst = output_root / row["split"] / row["label"] / row["filename"]
        if dst.exists():
            continue
        shutil.copy2(src, dst)


def check_leakage(manifest: pd.DataFrame) -> None:
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        va = set(manifest.loc[manifest["split"] == a, "video_id"])
        vb = set(manifest.loc[manifest["split"] == b, "video_id"])
        overlap = va & vb
        if overlap:
            raise RuntimeError(f"Leakage between {a} and {b}: {len(overlap)} videos, e.g. {list(overlap)[:5]}")


def build_stats(manifest: pd.DataFrame) -> dict:
    stats: dict = {"by_split": {}, "totals": {}}
    for split in ("train", "val", "test"):
        sub = manifest[manifest["split"] == split]
        stats["by_split"][split] = {
            "num_videos": int(sub["video_id"].nunique()),
            "num_sequences": len(sub),
            "num_fall_sequences": int((sub["label"] == "fall").sum()),
            "num_no_fall_sequences": int((sub["label"] == "no_fall").sum()),
        }
    stats["totals"] = {
        "num_videos": int(manifest["video_id"].nunique()),
        "num_sequences": len(manifest),
    }
    return stats




## 5. Dataloader & Models & Metrics & Export TFLite


In [ ]:
"""Load .npy skeleton sequences from split folders."""
from __future__ import annotations

import os
from glob import glob
from pathlib import Path

import numpy as np



def expected_shape() -> tuple[int, int]:
    return config.INPUT_TIMESTEPS, config.NUM_FEATURES


def load_dataset(
    data_path: str | Path,
    normalize: bool = True,
    min_confidence: float = config.MIN_KEYPOINT_CONFIDENCE,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    data_path = Path(data_path)
    exp_shape = expected_shape()
    x_list: list[np.ndarray] = []
    y_list: list[int] = []
    paths: list[str] = []

    print(f"Loading from {data_path}, expected shape {exp_shape}")
    for label_name, label_val in [("no_fall", 0), ("fall", 1)]:
        folder = data_path / label_name
        if not folder.is_dir():
            print(f"  Warning: missing folder {folder}")
            continue
        files = sorted(glob(str(folder / "*.npy")))
        loaded = 0
        for fp in files:
            try:
                arr = np.load(fp)
            except Exception as e:
                print(f"  Warning: cannot load {fp}: {e}")
                continue
            if arr.shape != exp_shape:
                print(f"  Warning: skip {fp} shape {arr.shape} != {exp_shape}")
                continue
            if normalize:
                arr = normalize_skeleton(arr, min_confidence=min_confidence)
                if np.isnan(arr).any():
                    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            x_list.append(arr.astype(np.float32))
            y_list.append(label_val)
            paths.append(fp)
            loaded += 1
        print(f"  {label_name}: {loaded}/{len(files)} sequences")

    if not x_list:
        return np.array([]), np.array([]), []
    return np.stack(x_list), np.array(y_list, dtype=np.float32), paths



In [ ]:
"""LSTM baseline for skeleton sequences."""
from __future__ import annotations

import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam



def create_lstm_classifier(
    input_shape: tuple[int, int] | None = None,
    lstm_units: int = config.LSTM_UNITS,
    dropout_rate: float = config.LSTM_DROPOUT,
    learning_rate: float = config.LEARNING_RATE,
) -> tf.keras.Model:
    if input_shape is None:
        input_shape = (config.INPUT_TIMESTEPS, config.NUM_FEATURES)

    f1_macro = tf.keras.metrics.F1Score(
        average="macro",
        threshold=0.5,
        name="f1_macro",
    )
    model = Sequential(name="lstm_fall_classifier")
    model.add(LSTM(lstm_units, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(dropout_rate))
    model.add(LSTM(lstm_units // 2))
    model.add(Dropout(dropout_rate))
    model.add(Dense(32, activation="relu"))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", f1_macro],
    )
    return model



In [ ]:
"""Transformer classifier for skeleton sequences."""
from __future__ import annotations

import tensorflow as tf
from tensorflow.keras.layers import (
    Add,
    Dense,
    Dropout,
    Embedding,
    GlobalAveragePooling1D,
    Input,
    LayerNormalization,
    MultiHeadAttention,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam



def transformer_encoder_block(
    inputs,
    d_model: int,
    num_heads: int,
    ff_dim: int,
    dropout_rate: float = 0.1,
    name_prefix: str = "",
):
    attn = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout_rate,
        name=f"{name_prefix}_mha",
    )(inputs, inputs, inputs)
    attn = Dropout(dropout_rate, name=f"{name_prefix}_mha_dropout")(attn)
    out1 = LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_layernorm1")(inputs + attn)

    ffn = Dense(ff_dim, activation="relu", name=f"{name_prefix}_ffn_dense1")(out1)
    ffn = Dense(d_model, name=f"{name_prefix}_ffn_dense2")(ffn)
    ffn = Dropout(dropout_rate, name=f"{name_prefix}_ffn_dropout")(ffn)
    out2 = LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_layernorm2")(out1 + ffn)
    return out2


def positional_embedding(seq_len: int, d_model: int, name_prefix: str = ""):
    positions = tf.range(start=0, limit=seq_len, delta=1)
    pos_2d = Embedding(
        input_dim=seq_len,
        output_dim=d_model,
        name=f"{name_prefix}_pos_embed",
    )(positions)
    return tf.expand_dims(pos_2d, axis=0)


def create_transformer_classifier(
    input_shape: tuple[int, int] | None = None,
    num_encoder_blocks: int = config.NUM_ENCODER_BLOCKS,
    d_model: int = config.D_MODEL,
    num_heads: int = config.NUM_HEADS,
    ff_dim: int = config.FF_DIM,
    projection_dim: int | None = None,
    final_dense_units: int = config.FINAL_DENSE_UNITS,
    dropout_rate: float = config.DROPOUT_RATE,
    learning_rate: float = config.LEARNING_RATE,
) -> Model:
    if input_shape is None:
        input_shape = (config.INPUT_TIMESTEPS, config.NUM_FEATURES)
    if projection_dim is None:
        projection_dim = d_model

    timesteps, _ = input_shape
    inputs = Input(shape=input_shape, name="input_features")
    x = Dense(projection_dim, name="feature_projection")(inputs)
    pos = positional_embedding(timesteps, projection_dim, name_prefix="pos_enc")
    x = Add(name="add_positional_encoding")([x, pos])
    x = Dropout(dropout_rate, name="input_dropout_after_pos_enc")(x)

    for i in range(num_encoder_blocks):
        x = transformer_encoder_block(
            x,
            projection_dim,
            num_heads,
            ff_dim,
            dropout_rate,
            name_prefix=f"encoder_block_{i + 1}",
        )

    x = GlobalAveragePooling1D(name="global_avg_pooling")(x)
    x = Dropout(dropout_rate, name="dropout_after_pooling")(x)
    x = Dense(final_dense_units, activation="relu", name="final_dense_1")(x)
    x = Dropout(dropout_rate / 2, name="dropout_final_dense")(x)
    outputs = Dense(1, activation="sigmoid", name="output_sigmoid")(x)

    model = Model(inputs=inputs, outputs=outputs)
    f1_macro = tf.keras.metrics.F1Score(
        average="macro",
        threshold=0.5,
        name="f1_macro",
    )
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", f1_macro],
    )
    return model



In [ ]:
"""Evaluation metrics and threshold tuning."""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    precision_recall_fscore_support,
)


@dataclass
class EvalResult:
    threshold: float
    accuracy: float
    precision_fall: float
    recall_fall: float
    f1_fall: float
    f2_fall: float
    confusion: np.ndarray
    report: str


def predict_labels(probs: np.ndarray, threshold: float) -> np.ndarray:
    return (probs.reshape(-1) >= threshold).astype(int)


def evaluate_at_threshold(
    y_true: np.ndarray,
    probs: np.ndarray,
    threshold: float,
) -> EvalResult:
    y_pred = predict_labels(probs, threshold)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average="binary", zero_division=0
    )
    f2 = fbeta_score(y_true, y_pred, beta=2, pos_label=1, zero_division=0)
    report = classification_report(
        y_true, y_pred, target_names=["no_fall", "fall"], zero_division=0
    )
    return EvalResult(
        threshold=threshold,
        accuracy=float(accuracy_score(y_true, y_pred)),
        precision_fall=float(prec),
        recall_fall=float(rec),
        f1_fall=float(f1),
        f2_fall=float(f2),
        confusion=cm,
        report=report,
    )


def find_best_threshold(
    y_true: np.ndarray,
    probs: np.ndarray,
    metric: str = "f2_fall",
    thresholds: np.ndarray | None = None,
) -> EvalResult:
    if thresholds is None:
        thresholds = np.arange(0.1, 0.91, 0.01)
    best: EvalResult | None = None
    for t in thresholds:
        result = evaluate_at_threshold(y_true, probs, float(t))
        score = getattr(result, metric)
        if best is None or score > getattr(best, metric):
            best = result
    assert best is not None
    return best


def error_analysis_df(
    y_true: np.ndarray,
    probs: np.ndarray,
    filepaths: list[str],
    threshold: float,
    split_name: str,
) -> pd.DataFrame:
    y_pred = predict_labels(probs, threshold)
    rows = []
    for i, fp in enumerate(filepaths):
        true_l = "fall" if y_true[i] == 1 else "no_fall"
        pred_l = "fall" if y_pred[i] == 1 else "no_fall"
        err_type = "TP" if y_true[i] == 1 and y_pred[i] == 1 else ""
        if y_true[i] == 0 and y_pred[i] == 1:
            err_type = "FP"
        elif y_true[i] == 1 and y_pred[i] == 0:
            err_type = "FN"
        elif y_true[i] == 0 and y_pred[i] == 0:
            err_type = "TN"
        rows.append(
            {
                "File Name": Path(fp).name,
                "True": true_l,
                "Pred": pred_l,
                "Prob": float(probs.reshape(-1)[i]),
                "Type": err_type,
                "Set": split_name,
            }
        )
    return pd.DataFrame(rows)


def metrics_summary_row(model_name: str, split_name: str, result: EvalResult) -> dict:
    return {
        "Model": model_name,
        "Split": split_name,
        "Threshold": result.threshold,
        "Accuracy": result.accuracy,
        "Precision_fall": result.precision_fall,
        "Recall_fall": result.recall_fall,
        "F1_fall": result.f1_fall,
        "F2_fall": result.f2_fall,
    }



In [ ]:
"""Export Keras model to TFLite."""
from __future__ import annotations

from pathlib import Path

import tensorflow as tf



def export_to_tflite(
    model: tf.keras.Model,
    export_dir: Path | None = None,
    tflite_path: Path | None = None,
) -> Path:
    export_dir = export_dir or (config.ARTIFACTS_DIR / config.SAVED_MODEL_DIR)
    tflite_path = tflite_path or (config.ARTIFACTS_DIR / config.TFLITE_MODEL_NAME)

    export_dir = Path(export_dir)
    tflite_path = Path(tflite_path)
    export_dir.mkdir(parents=True, exist_ok=True)
    tflite_path.parent.mkdir(parents=True, exist_ok=True)

    model.export(str(export_dir))
    converter = tf.lite.TFLiteConverter.from_saved_model(str(export_dir))
    tflite_bytes = converter.convert()
    tflite_path.write_bytes(tflite_bytes)
    size_kb = len(tflite_bytes) / 1024
    print(f"Saved TFLite: {tflite_path} ({size_kb:.2f} KB)")
    return tflite_path


def verify_tflite(tflite_path: Path) -> dict:
    interp = tf.lite.Interpreter(model_path=str(tflite_path))
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    info = {
        "input_shape": inp["shape"],
        "input_dtype": str(inp["dtype"]),
        "output_shape": out["shape"],
        "output_dtype": str(out["dtype"]),
    }
    expected = [1, config.INPUT_TIMESTEPS, config.NUM_FEATURES]
    if list(inp["shape"]) != expected:
        raise ValueError(f"TFLite input {inp['shape']} != expected {expected}")
    return info



## 6. Huấn luyện Mô hình (Training Logic)


In [ ]:
def train_model(
    model: tf.keras.Model,
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    y_val: np.ndarray,
    model_tag: str,
    epochs: int,
    batch_size: int,
) -> tf.keras.Model:
    callbacks = [
        EarlyStopping(
            monitor="val_f1_macro",
            patience=config.EARLY_STOPPING_PATIENCE,
            mode="max",
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_f1_macro",
            factor=0.5,
            patience=5,
            mode="max",
            min_lr=1e-6,
            verbose=1,
        ),
    ]
    y_train_2d = y_train.reshape(-1, 1)
    y_val_2d = y_val.reshape(-1, 1)
    history = model.fit(
        x_train,
        y_train_2d,
        validation_data=(x_val, y_val_2d),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1,
    )
    hist_path = config.ARTIFACTS_DIR / f"{model_tag}_history.json"
    hist_path.parent.mkdir(parents=True, exist_ok=True)
    serializable = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    hist_path.write_text(json.dumps(serializable, indent=2), encoding="utf-8")
    return model


## 7. CHẠY TOÀN BỘ PIPELINE (EXECUTION)
Thay đổi đường dẫn thư mục chứa video gốc của bạn vào 2 biến `RAW_VIDEO_FALL_DIR` và `RAW_VIDEO_NO_FALL_DIR` ở bên dưới.


In [ ]:

# TẢI DATASET TỪ KAGGLEHUB
import kagglehub
dataset_path = kagglehub.dataset_download("payutch/fall-video-dataset")
print("Path to dataset files:", dataset_path)

# Thư mục gốc của video
def find_video_dir(base_path, target_names):
    for p in Path(base_path).rglob('*'):
        if p.is_dir() and p.name.lower() in target_names:
            if any(p.glob('*.mp4')) or any(p.glob('*.avi')):
                return p
    return Path(base_path) / target_names[0]

RAW_VIDEO_FALL_DIR = find_video_dir(dataset_path, ["fall", "falls"])
RAW_VIDEO_NO_FALL_DIR = find_video_dir(dataset_path, ["no_fall", "nofall", "no_falls"])
print(f"Fall Video Dir: {RAW_VIDEO_FALL_DIR}")
print(f"No Fall Video Dir: {RAW_VIDEO_NO_FALL_DIR}")

if not (RAW_VIDEO_FALL_DIR.exists() and any(RAW_VIDEO_FALL_DIR.glob('*.mp4'))):
    print("\n[!] LỖI: KHÔNG THỂ TÌM THẤY VIDEO (.mp4) TRONG DATASET!")
    print("Cấu trúc thư mục dataset_path:")
    for root, dirs, files in os.walk(dataset_path):
        print(root, dirs, [f for f in files if f.endswith('.mp4')])
    print("\n")

config.DATA_RAW.mkdir(parents=True, exist_ok=True)
config.DATA_SPLITS.mkdir(parents=True, exist_ok=True)
config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
config.REPORT_DIR.mkdir(parents=True, exist_ok=True)

# 7.1 TRÍCH XUẤT ĐẶC TRƯNG TỪ VIDEO (Bỏ qua nếu đã có file .npy)
print("--- BƯỚC 1: TRÍCH XUẤT KHUNG XƯƠNG (MEDIAPIPE) ---")
if RAW_VIDEO_FALL_DIR.exists() and RAW_VIDEO_NO_FALL_DIR.exists():
    out_fall = config.DATA_RAW / "fall"
    out_nofall = config.DATA_RAW / "no_fall"
    out_fall.mkdir(parents=True, exist_ok=True)
    out_nofall.mkdir(parents=True, exist_ok=True)
    
    for vid in RAW_VIDEO_FALL_DIR.glob("*.mp4"):
        process_video(str(vid), "fall", out_fall, config.INPUT_TIMESTEPS, 15)
    for vid in RAW_VIDEO_NO_FALL_DIR.glob("*.mp4"):
        process_video(str(vid), "no_fall", out_nofall, config.INPUT_TIMESTEPS, 15)
else:
    print(f"CẢNH BÁO: Không tìm thấy thư mục video thô. Vui lòng kiểm tra lại đường dẫn!")

# 7.2 CHIA TẬP DỮ LIỆU
print("\n--- BƯỚC 2: CHIA TẬP DỮ LIỆU (VIDEO-LEVEL) ---")
try:
    records = collect_npy_files(config.DATA_RAW, "flat")
    video_label = assign_video_labels(records)
    video_ids = sorted(video_label.keys())
    labels = [video_label[v] for v in video_ids]
    assignment = split_videos(video_ids, labels, config.TRAIN_RATIO, config.VAL_RATIO, config.TEST_RATIO, config.SPLIT_RANDOM_STATE)
    
    rows = [{"split": assignment[r["video_id"]], **r} for r in records]
    manifest = pd.DataFrame(rows)
    check_leakage(manifest)
    manifest.to_csv(config.MANIFEST_PATH, index=False)
    
    copy_split_files(manifest, config.DATA_SPLITS, copy_files=True)
    print("Chia dữ liệu thành công! Đã sao chép vào data/splits.")
except Exception as e:
    print(f"Lỗi khi chia dữ liệu: {e}")

# 7.3 TRAIN MODEL
print("\n--- BƯỚC 3: HUẤN LUYỆN MÔ HÌNH ---")
x_train, y_train, _ = load_dataset(config.DATA_SPLITS / "train")
x_val, y_val, _ = load_dataset(config.DATA_SPLITS / "val")
x_test, y_test, test_paths = load_dataset(config.DATA_SPLITS / "test")

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

# Chọn 'lstm' hoặc 'transformer'
MODEL_TYPE = 'transformer'

if MODEL_TYPE == "lstm":
    model = create_lstm_classifier()
    model_tag = "lstm"
else:
    model = create_transformer_classifier()
    model_tag = "transformer"
    
keras_path = config.ARTIFACTS_DIR / f"{model_tag}_best.keras"
model = train_model(model, x_train, y_train, x_val, y_val, model_tag, config.EPOCHS, config.BATCH_SIZE)
model.save(keras_path)

val_probs = model.predict(x_val, verbose=0).reshape(-1)
best_val = find_best_threshold(y_val, val_probs, metric="f2_fall")
print(f"\nNgưỡng tốt nhất trên VAL: {best_val.threshold:.2f}")
print(best_val.report)

if len(x_test) > 0:
    test_probs = model.predict(x_test, verbose=0).reshape(-1)
    test_res = evaluate_at_threshold(y_test, test_probs, best_val.threshold)
    print(f"\nKẾT QUẢ TEST @ threshold {best_val.threshold:.2f}:")
    print(test_res.report)

if MODEL_TYPE == "transformer":
    tflite_path = export_to_tflite(model)
    verify_tflite(tflite_path)
    print(f"TFLite exported successfully to: {tflite_path}")

